# Iterators and generators

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-04-iterators-and-generators.ipynb)

Data engineering works on data that does not fit in memory. A list of ten million rows needs gigabytes, but a generator that yields one row at a time needs almost nothing. This chapter explains how Python's iteration protocol works, how to write generators, when they run, and how the standard library `itertools` module combines them into pipelines.

Every rule is stated before the code that shows it. Each code cell's `assert` lines are the expected results. If an `assert` fails, your model of that rule is wrong.

## Iterables and iterators

An **iterable** is anything a `for` loop can walk: a list, a string, a dict, a file, a `range`. An **iterator** is the object that does the walking. You get one by calling `iter()` on an iterable. An iterator has `next()`, which returns the following item, and raises `StopIteration` when nothing is left.

Two rules follow from this:

- A **list is an iterable**. Each `for` loop calls `iter()` on it again, so you can loop over a list many times.
- An **iterator is single-use**. Once `next()` has returned an item, that item is gone. A second loop over the same iterator sees only what is left, which may be nothing.

### Where you'll use it

- **Reading files and database cursors:** a file object, a DB-API cursor and a Kafka consumer are all iterators. A `for` loop over them reads one record at a time, so memory stays flat.
- **Feeding APIs:** a paginated REST API is an iterator over pages. The loop asks for the next page only when it needs it.
- **Why it matters:** code written for a list also works on an iterator. The same transform runs on a 1 KB test file and on a 50 GB production file.

In [ ]:
numbers = [10, 20, 30]
it = iter(numbers)
assert next(it) == 10
assert next(it) == 20
assert next(it) == 30

try:
    next(it)
except StopIteration:
    pass
else:
    raise AssertionError("expected StopIteration")

# next() accepts a default, which avoids StopIteration on an empty iterator.
assert next(iter([]), "empty") == "empty"

# A list can be looped many times. An iterator can be looped only once.
assert sum(numbers) == 60 and sum(numbers) == 60
single_use = iter(numbers)
assert sum(single_use) == 60
assert sum(single_use) == 0

## Writing your own iterable

Any class with `__iter__` is iterable, and any class with `__next__` is an iterator. The usual pattern is an iterable whose `__iter__` returns a **new** iterator each time. Then the object can be looped many times, like a list, while each loop keeps its own position.

In [ ]:
class CountdownIterator:
    """One pass over the countdown. It keeps its own position."""

    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

class Countdown:
    """Iterable: every loop gets a fresh CountdownIterator."""

    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)

countdown = Countdown(3)
assert list(countdown) == [3, 2, 1]
assert list(countdown) == [3, 2, 1]          # reusable, because each loop starts fresh

## Generators: `yield`

A function that contains `yield` is a **generator function**. Calling it does not run its body. It returns a **generator**, which is an iterator. Each `next()` runs the body until the next `yield`, hands that value to the caller, and pauses. When the body finishes, the generator raises `StopIteration`, and it yields nothing more.

Because a generator computes each value on demand, it is **lazy**. Nothing in its body runs until someone asks for a value.

### Where you'll use it

- **Streaming ETL:** a generator reads, parses and yields one cleaned record at a time into a bulk writer.
- **Paginated APIs:** a generator fetches a page, yields its items, then fetches the next page only when the caller needs it.
- **Live feeds:** a generator that tails a log file and yields each new line as it is appended.

In [ ]:
def read_batches(total, size):
    start = 0
    while start < total:
        yield list(range(start, min(start + size, total)))
        start += size

batches = read_batches(7, 3)
assert next(batches) == [0, 1, 2]
assert list(batches) == [[3, 4, 5], [6]]
assert list(batches) == []                   # an exhausted generator yields nothing more

In [ ]:
log = []

def numbers_with_log(n):
    for i in range(n):
        log.append(f"produce {i}")
        yield i

gen = numbers_with_log(2)
assert log == []                             # calling the function ran none of its body
assert next(gen) == 0
assert log == ["produce 0"]                  # only the first step has run
assert list(gen) == [1]
assert log == ["produce 0", "produce 1"]

### `yield from`: delegate to another iterable

`yield from iterable` yields every item of the iterable, one at a time, as if the loop were written out. It is the standard way to flatten a generator of generators or of lists.

In [ ]:
def pages():
    yield [1, 2]
    yield [3]

def flatten_pages(page_iter):
    for page in page_iter:
        yield from page

assert list(flatten_pages(pages())) == [1, 2, 3]

## Generator expressions

A generator expression looks like a list comprehension with parentheses. It is lazy, like any generator. Use it when you loop over the values once, or when you pass the result straight to a function such as `sum`. Use a list comprehension when you need to index the result, read its length or loop over it twice.

### Where you'll use it

- **Aggregating large inputs:** `sum(row.amount for row in rows if row.is_valid)` computes a total without building a list.
- **Feeding writers:** a generator expression passed to `csv.writer.writerows` writes millions of rows with constant memory.

In [ ]:
amounts = range(1, 1_000_001)

total = sum(a for a in amounts if a % 2 == 0)       # no list is built
assert total == 250_000_500_000

gen_expr = (a * a for a in range(5))
assert not isinstance(gen_expr, list)
assert sum(gen_expr) == 30
assert sum(gen_expr) == 0                           # one pass only, like any iterator

squares = [a * a for a in range(5)]                 # a list: indexable and reusable
assert squares[-1] == 16

## A streaming pipeline

Chain generators so that each stage pulls one record from the previous stage. The records move through the chain one at a time, so the whole input never sits in memory. This is the in-memory shape of most ETL jobs: read, parse, filter, transform.

Two rules matter here. A stage runs only when the last stage asks for a value, so the whole chain runs lazily. And a record that a filter drops disappears silently, so a stage that filters should count what it drops, as the next chapter's quarantine pattern does.

### Where you'll use it

- **Multi-stage transforms:** a chain of parse, validate and enrich stages is the shape of most batch ETL jobs. Each stage can be tested alone.
- **Back-pressure:** each stage pulls from the one before it, so a slow writer slows the reader instead of filling memory.

In [ ]:
raw_lines = [
    "ana,30,ok",
    "ben,x,ok",
    "cy,41,ok",
    "dee,25,error",
]

def parse(lines):
    for line in lines:
        name, age, status = line.split(",")
        yield {"name": name, "age": age, "status": status}

def valid_age(records):
    for record in records:
        if record["age"].isdigit():           # "x" is dropped here, silently
            record["age"] = int(record["age"])
            yield record

def only_ok(records):
    return (r for r in records if r["status"] == "ok")

clean = list(only_ok(valid_age(parse(raw_lines))))
assert [r["name"] for r in clean] == ["ana", "cy"]
assert clean[1]["age"] == 41

## `itertools`: standard building blocks

`itertools` provides fast, memory-light functions that return iterators. The ones you will use most:

- `count(start)` is endless. Cap it with `islice`.
- `islice(it, n)` takes the first `n` items from any iterator, including an endless one.
- `chain(a, b)` joins iterables without copying them. `chain.from_iterable(x)` joins the items of `x`.
- `accumulate(values, func)` yields running results. It defaults to a running sum.
- `groupby(items, key)` groups **consecutive** items with the same key. It does not sort. Sort by the key first when you need every group in one piece.
- `zip_longest(a, b, fillvalue=...)` pairs items until the **longer** input ends.
- `product(a, b)` gives every combination, as nested loops would.

Each group from `groupby` is an iterator that becomes invalid as soon as `groupby` moves on. Consume it before the next step.

### Where you'll use it

- **Taking a sample:** `islice` reads the first N rows of a huge file to inspect its schema.
- **Sessionising or grouping sorted events:** `groupby` on a user ID, after sorting by `(user_id, timestamp)`.
- **Joining streams:** `chain` concatenates the output of several daily files into one iterator.

In [ ]:
from itertools import accumulate, chain, count, groupby, islice, product, repeat, zip_longest

assert list(islice(count(1), 3)) == [1, 2, 3]
first_five_odd = list(islice((n for n in count(1) if n % 2), 5))
assert first_five_odd == [1, 3, 5, 7, 9]

assert list(chain([1, 2], [3], [])) == [1, 2, 3]
assert list(chain.from_iterable([[1], [2, 3]])) == [1, 2, 3]
assert list(accumulate([1, 2, 3, 4])) == [1, 3, 6, 10]
assert list(accumulate([3, 1, 4], max)) == [3, 3, 4]         # a running maximum
assert list(zip_longest([1, 2], ["a"], fillvalue="-")) == [(1, "a"), (2, "-")]
assert list(product([1, 2], "ab")) == [(1, "a"), (1, "b"), (2, "a"), (2, "b")]
assert list(repeat("x", 3)) == ["x", "x", "x"]

In [ ]:
# groupby only merges neighbours. The two "a" runs stay separate.
runs = [(key, len(list(group))) for key, group in groupby("aabbba")]
assert runs == [("a", 2), ("b", 3), ("a", 1)]

events = [("ana", 1), ("ana", 2), ("ben", 5), ("ana", 3)]
events.sort(key=lambda e: e[0])               # sort first, so each user is one run
groups = {name: [v for _, v in items] for name, items in groupby(events, key=lambda e: e[0])}
assert groups == {"ana": [1, 2, 3], "ben": [5]}

## Chunking an iterator

Batching a stream into fixed-size chunks is one of the most common pipeline steps. It bounds memory, and it gives the database a sensible write size. `itertools.batched` does this from Python 3.12. The version below works on any Python 3 release. It takes `islice` on the same iterator again and again, and stops when a chunk comes back empty.

### Where you'll use it

- **Bulk database writes:** `executemany` or a `COPY` statement is much faster per batch of 1,000 rows than per row.
- **API limits:** sending 100 records per request to stay under a provider's payload limit.
- **Parallel work units:** splitting a stream into chunks that a worker pool processes at the same time.

In [ ]:
def chunked(iterable, size):
    if size < 1:
        raise ValueError("size must be at least 1")
    iterator = iter(iterable)
    while True:
        chunk = list(islice(iterator, size))
        if not chunk:
            return
        yield chunk

assert list(chunked(range(5), 2)) == [[0, 1], [2, 3], [4]]
assert list(chunked([], 3)) == []

try:
    list(chunked([1], 0))
except ValueError:
    pass
else:
    raise AssertionError("expected ValueError")

## Practice questions

1. Write `running_max(values)` as a generator that yields the largest value seen so far.
2. Write `count_lines_longer_than(lines, limit)` that uses a generator expression and returns a count.
3. Use `groupby` to compress a list of states into `(state, run_length)` pairs. For example, `["up", "up", "down"]` gives `[("up", 2), ("down", 1)]`.

## Answers

### Question 1

In [ ]:
def running_max(values):
    best = None
    for value in values:
        best = value if best is None or value > best else best
        yield best

assert list(running_max([3, 1, 4, 1, 5])) == [3, 3, 4, 4, 5]

### Question 2

In [ ]:
def count_lines_longer_than(lines, limit):
    return sum(1 for line in lines if len(line) > limit)

assert count_lines_longer_than(["a", "abcd", "abc"], 2) == 2

### Question 3

In [ ]:
from itertools import groupby

def run_lengths(states):
    return [(state, sum(1 for _ in run)) for state, run in groupby(states)]

assert run_lengths(["up", "up", "down"]) == [("up", 2), ("down", 1)]
assert run_lengths(["up", "down", "up"]) == [("up", 1), ("down", 1), ("up", 1)]

## Interview notes

- **Iterators are single-pass.** A second loop over the same generator sees nothing. Store the result, or create the generator again, when you need two passes.
- **Generators are lazy.** Nothing in a generator body runs until a consumer asks for a value. A bug in a middle stage appears only when the last stage pulls.
- **Iterables are reusable; iterators are not.** A class whose `__iter__` returns a new iterator can be looped many times.
- **`groupby` groups consecutive items only.** Sort first for full groups. When the data does not fit in memory, sorting needs an external sort, and grouping belongs in a database `GROUP BY` or an engine such as Spark.
- **Stream, do not load.** Read and transform with generators, and hold only one batch at a time.